In [1]:
import os 
import pwd
import json
import pandas as pd
import numpy as np
import sqlalchemy as sa

from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sqlalchemy import text

from IPython.display import display, HTML
import statistics
from sklearn.impute import SimpleImputer

from datetime import date
from datetime import datetime

# import xgboost
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pyodbc
import pandas as pd

# fn to setup SQL connectivity
def get_sql_connection():
    import pwd
    import json
    import os

    # get the current RHEL user running this script 
    clientusr=pwd.getpwuid(os.getuid())[0]
    connection_string_json='/datafs_a/SQL/cstring'

    with open(connection_string_json) as f:
        data = json.load(f)
        srv = data['S']
        un  = data['U']
        pw  = data['P']
        db  = data['D']

    # Build unencoded connection string (for pyodbc directly)
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={srv},1433;"
        f"DATABASE={db};"
        f"UID={un};"
        f"PWD={pw};"
        f"APP={clientusr};"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
    )

    return pyodbc.connect(conn_str)
	
def send_query(sql_query):

    cursor.execute(sql_query)

    # Fetch column names
    columns = [column[0] for column in cursor.description]

    # Fetch all rows
    rows = cursor.fetchall()

    # Ensure rows are converted properly (each row is a tuple)
    df = pd.DataFrame.from_records([tuple(row) for row in rows], columns=columns)

    return df

## Main ##

#get a connection to mssql on HHCMITSQL001
conn = get_sql_connection()
cursor = conn.cursor()


In [13]:
import random
random.seed(10)

In [14]:
base = pd.read_csv('/datafs_a/carolgao/haim14_data/blood_infection/all_blood_cultures_unique_encounter.csv')

In [17]:
# sort on patientkey keep only one patient key if same encounter and same lab
base = base.sort_values('PatientDurableKey', ascending = False).drop_duplicates(['EncounterKey', 'CollectionInstant_Rounded'])
# check unique on order time and encounter key
len(base) == len(base.drop_duplicates(subset=['CollectionInstant_Rounded','EncounterKey']))

True

In [18]:
# Extract ordered time within a day
base['CollectionInstant_Rounded'] = base['CollectionInstant_Rounded'].astype(str)
base['CollectionInstant_Rounded'] = base['CollectionInstant_Rounded'].str[11:]
base['CollectionInstant_Rounded'] = base['CollectionInstant_Rounded'].str.replace(':', '')
base['CollectionInstant_Rounded'] = base['CollectionInstant_Rounded'].str[:4]
base['OrderedTimeOfDay'] = pd.to_numeric(base['CollectionInstant_Rounded'])
base = base.drop(columns=['CollectionInstant_Rounded'])

In [19]:
base['Positive'].mean()

np.float64(0.08928429476138743)

### Vitals 

In [20]:
vitals_raw = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/vitals_raw_all.csv")

In [21]:
spo2_raw = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/spo2_raw_all.csv")
pulse_raw = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/pulse_raw_all.csv")

In [22]:
vitals = pd.concat([vitals_raw, spo2_raw, pulse_raw])

In [23]:
# Merge with cleaned cases
vitals = vitals[['PatientDurableKey', 'EncounterKey', 'FlowsheetRowKey', 'DateKey', 'TimeOfDayKey', 'Value', 'NumericValue', 'DisplayName', 'ValueType']]
vitals = vitals.merge(base, how='left', on=['PatientDurableKey', 'EncounterKey']) 

In [24]:
len(vitals['EncounterKey'].unique())

149890

In [25]:
# Vital time needs to be strictly before the culture was ordered
vitals = vitals.loc[(vitals['DateKey'] < vitals['OrderedDateKey']) | 
           ((vitals['DateKey'] == vitals['OrderedDateKey']) 
           & (vitals['TimeOfDayKey'] <= vitals['OrderedTimeOfDay']))]

In [26]:
len(vitals['EncounterKey'].unique()) # 99258

145135

In [27]:
# drop missings 
vitals = vitals.loc[vitals['NumericValue'].isnull() == False]

In [28]:
vital_counts = pd.Series(vitals['DisplayName']).value_counts()
for n, c in vital_counts.items():
    print(n, c)

Pulse 1572808
SpO2 1560667
Resp 1360053
Temp 880074
SPO2 15470
SpO2 Alarm Limit Low 4583
SpO2 Alarm Limit High 2998
SpO2: Pre-Ductal (Right Hand) 1338
Heart Rate 1066
Treatment SpO2 (%) 345
SpO2: Post-Ductal (Left Hand/Foot) 329
Pretreatment SpO2 (%) 327
Ambulatory SpO2 299
Resting SpO2 266
Event SpO2 244
Posttreatment SpO2 (%) 148
Supplemental Oxygen Required for SpO2 =92% 127
SpO2: Pre-Ductal (CCHD) 97
SpO2 Difference (CCHD) 96
SpO2: Post-Ductal (CCHD) 96
SpO2 Low Limit 26
Minutes of SpO2 < 88% recorded 25
Minimum SpO2% 21
SpO2 Prior to Speaking Valve Placement 18
SpO2 During Speaking Valve Placement 16
SpO2 After Speaking Valve Placement 13
2022/09 RETIRED SpO2 Level, Food and Liquid Trials (NIS) 5
Initial SpO2 2
Final SpO2 2
SpO2 High Limit 2
2021/07/06 RETIRED Post SpO2 (%) 1
2021/07/06 RETIRED Pre SpO2 (%) 1


In [29]:
# convert to lower case to align 
vitals['DisplayName'] = vitals['DisplayName'].str.lower()

In [30]:
vitals = vitals.loc[vitals['DisplayName'].str.contains('minutes') == False]
vitals = vitals.loc[vitals['DisplayName'] != "supplemental oxygen required for spo2 =92%"]
vitals = vitals.loc[vitals['DisplayName'] != "spo2 difference (cchd)"]

In [31]:
# Keep the latest oxygen
spo2 = vitals.loc[(vitals['DisplayName'].str.contains('spo2'))]
#spo2 = spo2.loc[(spo2['DisplayName'] == "spo2") | (spo2['DisplayName'] == "ambulatory spo2") | (spo2['DisplayName'] == "resting spo2")
#| (spo2['DisplayName'] == "spo2 alarm limit low") | (spo2['DisplayName'] == "spo2 alarm limit high")]                                                                                              
spo2['latest_time'] = spo2.groupby(['PatientDurableKey','EncounterKey', 'DateKey'])['TimeOfDayKey'].transform(max)
spo2['latest_day'] = spo2.groupby(['PatientDurableKey','EncounterKey'])['DateKey'].transform(max)
spo2 = spo2.loc[(spo2['latest_time'] == spo2['TimeOfDayKey']) & (spo2['latest_day'] == spo2['DateKey'])]
print(len(spo2))

144845


In [32]:
# Keep the latest respiratory 
resp = vitals.loc[(vitals['DisplayName'] == 'resp')]
resp['latest_time'] = resp.groupby(['PatientDurableKey','EncounterKey', 'DateKey'])['TimeOfDayKey'].transform(max)
resp['latest_day'] = resp.groupby(['PatientDurableKey','EncounterKey'])['DateKey'].transform(max)
resp = resp.loc[(resp['latest_time'] == resp['TimeOfDayKey']) & (resp['latest_day'] == resp['DateKey'])]
print(len(resp))

143801


In [33]:
# Keep the latest heart rate 
pulse = vitals.loc[(vitals['DisplayName'] == 'pulse') | (vitals['DisplayName'] == 'herat rate')]
pulse['latest_time'] = pulse.groupby(['PatientDurableKey','EncounterKey', 'DateKey'])['TimeOfDayKey'].transform(max)
pulse['latest_day'] = pulse.groupby(['PatientDurableKey','EncounterKey'])['DateKey'].transform(max)
pulse = pulse.loc[(pulse['latest_time'] == pulse['TimeOfDayKey']) & (pulse['latest_day'] == pulse['DateKey'])]
print(len(pulse))

145075


In [34]:
# Keep the latest temperature
temp = vitals.loc[(vitals['DisplayName'] == 'temp')]
temp['latest_time'] = temp.groupby(['PatientDurableKey','EncounterKey', 'DateKey'])['TimeOfDayKey'].transform(max)
temp['latest_day'] = temp.groupby(['PatientDurableKey','EncounterKey'])['DateKey'].transform(max)
temp = temp.loc[(temp['latest_time'] == temp['TimeOfDayKey']) & (temp['latest_day'] == temp['DateKey'])]
print(len(temp))

140875


In [35]:
# Keep the highest value if two values at the same time on the same day
spo2['SpO2'] = spo2.groupby(['PatientDurableKey','EncounterKey'])['NumericValue'].transform(max)
spo2 = spo2.drop_duplicates(subset=['PatientDurableKey', 'EncounterKey', 'SpO2'])[['PatientDurableKey','EncounterKey','SpO2']]
resp['Resp'] = resp.groupby(['PatientDurableKey','EncounterKey'])['NumericValue'].transform(max)
resp = resp.drop_duplicates(subset=['PatientDurableKey', 'EncounterKey', 'Resp'])[['PatientDurableKey','EncounterKey','Resp']]
pulse['Pulse'] = pulse.groupby(['PatientDurableKey','EncounterKey'])['NumericValue'].transform(max)
pulse = pulse.drop_duplicates(subset=['PatientDurableKey', 'EncounterKey', 'Pulse'])[['PatientDurableKey','EncounterKey','Pulse']]
temp['Temp'] = temp.groupby(['PatientDurableKey','EncounterKey'])['NumericValue'].transform(max)
temp = temp.drop_duplicates(subset=['PatientDurableKey', 'EncounterKey', 'Temp'])[['PatientDurableKey','EncounterKey','Temp']]

In [36]:
# Merge in oxygen level, resp rate, and heart rate. Left merge so that we can have NAs
final_cases = pd.merge(base, spo2, how="left", on=['PatientDurableKey','EncounterKey'])
final_cases = pd.merge(final_cases, resp, how="left", on=['PatientDurableKey','EncounterKey'])
final_cases = pd.merge(final_cases, pulse, how="left", on=['PatientDurableKey','EncounterKey'])
final_cases = pd.merge(final_cases, temp, how="left", on=['PatientDurableKey','EncounterKey'])

In [37]:
# drop cases without vitals
final_cases = final_cases[(final_cases['SpO2'].isnull() == False) & (final_cases['Resp'].isnull() == False) & (final_cases['Pulse'].isnull() == False) & (final_cases['Temp'].isnull() == False)]

In [38]:
len(final_cases)

139015

In [39]:
# # Create threshold variables for vitals
# THE ASTYPES HAVE NOT BEEN TESTED (manually added for current Base)
final_cases['HighFever'] = final_cases['Temp'].apply(lambda x: np.nan if pd.isna(x) else x > 100.4).astype('Int64')
final_cases['LowFever']  = final_cases['Temp'].apply(lambda x: np.nan if pd.isna(x) else x < 96).astype('Int64')
final_cases['HighPulse'] = final_cases['Pulse'].apply(lambda x: np.nan if pd.isna(x) else x > 90).astype('Int64')
final_cases['LowOxygen'] = final_cases['SpO2'].apply(lambda x: np.nan if pd.isna(x) else x < 92).astype('Int64')

In [40]:
final_cases.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/final_vitals.csv", index=False)

In [41]:
statistics.mean(final_cases['Positive'])

0.08854440168327159

In [42]:
final_cases.groupby(['OrderedYear']).agg(
    positive_rate=('Positive', 'mean'),
    N = ('Positive', 'count')
)

,positive_rate,N
OrderedYear,,
2022,0.087535,41218
2023,0.085649,41997
2024,0.090750,36639
2025,0.092845,19161


### Historical Diagnoses

In [43]:
unique_patients = ",".join([str(x) for x in final_cases['PatientDurableKey'].unique()])

In [44]:
%%time
hist_diag = send_query(f"select distinct d1.PatientDurableKey, d1.EncounterKey, d1.StartDateKey, d1.EndDateKey, d1.DiagnosisKey, d1.Type \
from CAB_EPIC.dbo.DiagnosisEventFact d1 \
where d1.DiagnosisKey > 0 and d1.Type = 'Billing Diagnosis' and d1.PatientDurableKey in ({unique_patients}) ")

CPU times: user 1min 1s, sys: 5.54 s, total: 1min 6s
Wall time: 6min 39s


In [46]:
len(hist_diag['PatientDurableKey'].unique())

88992

### ICD Diagnoses

In [47]:
icd_group = send_query("select distinct DiagnosisKey, GroupedNameAndCode from CAB_EPIC.dbo.DiagnosisTerminologyDim where GroupedNameandCode != '' and GroupedNameandCode != '*Deleted' ")

In [48]:
# Merge in Grouped ICD for each of our cohort patient's historical diagnoses
diagnosis_icd = hist_diag.merge(icd_group, left_on='DiagnosisKey', right_on='DiagnosisKey')
# Rename EncounterKey in ICD table to history encounters to avoid confusion with encounter keys in our cohort cases
diagnosis_icd = diagnosis_icd.rename(columns={"EncounterKey": "Hist_Encounter"})

In [49]:
# Split into names and ICD codes
diagnosis_icd[['ICDname', 'ICDcode']] = diagnosis_icd['GroupedNameAndCode'].str.split(': ', expand=True)

In [51]:
diagnosis_icd.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/diagnoses_raw.csv", index = False)

In [65]:
diagnosis_icd = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/diagnoses_raw.csv")

In [68]:
# Convert to chapters 
def convert_to_chapter(code):
    if code.startswith("A") or code.startswith("B"): 
        return "1 Certain infectious and parasitic diseases"
    elif code.startswith("C") or (code.startswith("D") and code[1:3].isdigit() and int(code[1:3]) <= 48):
        return "2 Neoplasms" 
    elif code.startswith("D") and code[1:3].isdigit() and int(code[1:3]) >= 50 and int(code[1:3]) <= 89: 
        return "3 Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism"
    elif code.startswith("E") and code[1:3].isdigit() and int(code[1:3]) >= 0 and int(code[1:3]) <= 90: 
        return "4 Endocrine, nutritional and metabolic diseases"
    elif code.startswith("F"): 
        return "5 Endocrine, nutritional and metabolic diseases"
    elif code.startswith("G"): 
        return "6 Diseases of the nervous system"
    elif code.startswith("H") and code[1:3].isdigit() and int(code[1:3]) >=0 and int(code[1:3]) <= 59:
        return "7 Diseases of the eye and adnexa" 
    elif code.startswith("H") and code[1:3].isdigit() and int(code[1:3]) >= 60 and int(code[1:3]) <= 95: 
        return "8 Diseases of the ear and mastoid process"
    elif code.startswith("I"):
        return "9 Diseases of the circulatory system"
    elif code.startswith("J"): 
        return "10 Diseases of the respiratory system"
    elif code.startswith("K") and code[1:3].isdigit() and int(code[1:3]) >= 0 and int(code[1:3]) <= 93:
        return "11 Diseases of the respiratory system"
    elif code.startswith("L"): 
        return "12 Diseases of the respiratory system"
    elif code.startswith("M"):
        return "13 Diseases of the musculoskeletal system and connective tissue"
    elif code.startswith("N"):
        return "14 Diseases of the genitourinary system"
    elif code.startswith("O"): 
        return "15 Pregnancy, childbirth and the puerperium"
    elif code.startswith("P") and code[1:3].isdigit() and int(code[1:3]) >=0 and int(code[1:3]) <= 96:
        return "16 Certain conditions originating in the perinatal period"
    elif code.startswith("Q"): 
        return "17 Congenital malformations, deformations and chromosomal abnormalities"
    elif code.startswith("R"): 
        return "18 Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified"
    elif code.startswith("S") or code.startswith("T"): 
        return "19 Injury, poisoning and certain other consequences of external causes"
    elif code.startswith("V") or code.startswith("W") or code.startswith("X") or code.startswith("Y"):
        return "20 External causes of morbidity and mortality"
    elif code.startswith("Z"):
        return "21 Factors influencing health status and contact with health services"
    elif code.startswith("U"):
        return "22 Codes for special purposes" 

In [69]:
def past_infection(code):
    if code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 0 and int(code[1:3]) <= 9:
        return "Intestinal infectious diseases"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 15 and int(code[1:3]) <= 19:
        return "Tuberculosis"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 20 and int(code[1:3]) <= 28:
        return "Certain zoonotic bacterial diseases"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 30 and int(code[1:3]) <= 49:
        return "Other bacterial diseases"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 50 and int(code[1:3]) <= 64:
        return "Infections with a predominantly sexual mode of transmission"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 65 and int(code[1:3]) <= 69:
        return "Other spirochaetal diseases"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 70 and int(code[1:3]) <= 74:
        return "Other diseases caused by chlamydiae"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 75 and int(code[1:3]) <= 79:
        return "Rickettsioses"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 80 and int(code[1:3]) <= 89:
        return "Viral infections of the central nervous system"
    elif code.startswith("A") and code[1:3].isdigit() and int(code[1:3]) >= 92 and int(code[1:3]) <= 99:
        return "Arthropod-borne viral fevers and viral haemorrhagic fevers"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 0 and int(code[1:3]) <= 9:
        return "Viral infections characterized by skin and mucous membrane lesions"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 15 and int(code[1:3]) <= 19:
        return "Viral hepatitis"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 20 and int(code[1:3]) <= 24:
        return "Human immunodeficiency virus HIV disease"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 25 and int(code[1:3]) <= 34:
        return "Other viral diseases"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 35 and int(code[1:3]) <= 49:
        return "Mycoses"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 50 and int(code[1:3]) <= 64:
        return "Protozoal diseases"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 65 and int(code[1:3]) <= 83:
        return "Helminthiases"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 85 and int(code[1:3]) <= 89:
        return "Pediculosis, acariasis and other infestations"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 90 and int(code[1:3]) <= 94:
        return "Sequelae of infectious and parasitic diseases"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 95 and int(code[1:3]) <= 98:
        return "Bacterial, viral and other infectious agents"
    elif code.startswith("B") and code[1:3].isdigit() and int(code[1:3]) >= 99:
        return "Other infectious diseases"
    elif code.startswith("J") and code[1:3].isdigit() and int(code[1:3]) >= 0 and int(code[1:3]) <= 22:
        return "Acute respiratory infections"
    elif code.startswith("Z") and code[1:3].isdigit() and int(code[1:3]) == 22:
        return "Carrier of infectious disease"
    elif code.startswith("P") and code[1:3].isdigit() and int(code[1:3]) >= 35 and int(code[1:3]) <= 39:
        return "Infectious and parasitic diseases specific to the perinatal period"
    elif code.startswith("O") and code[1:3].isdigit() and int(code[1:3]) == 98:
        return "Infectious and parasitic diseases complicating pregnancy, childbirth and the puerperium"

In [70]:
# group by chapters 
diagnosis_icd['ICDchapter'] = diagnosis_icd['ICDcode'].apply(convert_to_chapter)
diagnosis_icd['ICD_infection'] = diagnosis_icd['ICDcode'].apply(past_infection)

In [71]:
# keep more granular levels for infection-related diagnoses
diagnosis_icd.loc[(diagnosis_icd['ICDchapter'] == "1 Certain infectious and parasitic diseases") & (diagnosis_icd['ICD_infection'].isnull() == False) , 'ICDchapter'] = diagnosis_icd.loc[diagnosis_icd['ICDchapter'] == "1 Certain infectious and parasitic diseases", 'ICD_infection']

In [72]:
diagnosis_icd = diagnosis_icd[['PatientDurableKey', 'Hist_Encounter', 'StartDateKey', 'EndDateKey', 'ICDchapter', 'ICDname', 'ICDcode']]

In [73]:
# Restrict to cases that belong to each patient's previous encounters
history_merged = diagnosis_icd.merge(final_cases, left_on='PatientDurableKey', right_on='PatientDurableKey')
cleaned_ICD = history_merged.loc[(history_merged['Hist_Encounter'] != history_merged['EncounterKey']) 
                   & (history_merged['EndDateKey'] < history_merged['OrderedDateKey'])]

In [76]:
# Pivot table for chapters
cleaned_ICD_chapter = cleaned_ICD[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay','ICDchapter']]
cleaned_ICD_chapter = cleaned_ICD_chapter.pivot_table(index=['PatientDurableKey','EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], columns='ICDchapter', aggfunc=len, fill_value=0)

In [77]:
# Pivot table for individual icd's
cleaned_ICD_name = cleaned_ICD[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay','ICDname']]
cleaned_ICD_name = cleaned_ICD_name.pivot_table(index=['PatientDurableKey','EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], columns='ICDname', aggfunc=len, fill_value=0)

In [78]:
cleaned_ICD_chapter = cleaned_ICD_chapter.add_prefix('chapter_icd_')
cleaned_ICD_name = cleaned_ICD_name.add_prefix('icd_')

In [79]:
# Merge with case diagnoses (tabular format)
final_cases = final_cases.merge(cleaned_ICD_chapter, how = "left", on=['PatientDurableKey','EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'])
final_cases = final_cases.merge(cleaned_ICD_name, how = "left", on=['PatientDurableKey','EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'])

In [81]:
len(final_cases) # 95610

139015

In [82]:
#fill 0 for those with no historical ICD 
icd_columns = [c for c in final_cases.columns if (c.startswith('icd_'))]
final_cases[icd_columns] = final_cases[icd_columns].fillna(value=0)
chapter_columns = [c for c in final_cases.columns if (c.startswith('chapter_'))]
final_cases[chapter_columns] = final_cases[chapter_columns].fillna(value=0)

In [83]:
final_cases.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/base_vitals_icd.csv", index=False)

### Chief Complaints

In [4]:
final_cases = pd.read_csv("/datafs_a/haim14_data/blood_infection/base_vitals_icd.csv")

In [84]:
%%time
complaints = send_query("SELECT ChiefComplaintKey, EncounterKey AS CompEncounterKey, PatientDurableKey FROM CAB_EPIC.dbo.ChiefComplaintEventFact WHERE (PatientKey != -3) AND (PatientKey != -1)")

complaints = complaints.merge(send_query("SELECT ChiefComplaintKey, Name As CompName, Abbreviation as CompAbbr FROM CAB_EPIC.dbo.ChiefComplaintDim"),
                                how='inner', left_on='ChiefComplaintKey', right_on='ChiefComplaintKey')

CPU times: user 1min 13s, sys: 12.9 s, total: 1min 26s
Wall time: 1min 27s


In [85]:
# Merging on encounter key (so do not consider prior chief complaints)
patients_cmp = final_cases[['EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay']].merge(complaints, how='left', left_on='EncounterKey', right_on='CompEncounterKey')

In [87]:
# Pivot on encounter key, since duplicate patients, but if from same encounter chief complaint should be same regardless
patients_cmp = patients_cmp[['EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'CompName']]
patients_cmp = patients_cmp.pivot_table(index=['EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], columns='CompName', aggfunc=len, fill_value=0)

In [88]:
# Add prefix 
patients_cmp = patients_cmp.add_prefix('cmp_')

In [89]:
# Merge with cases
final_cases = final_cases.merge(patients_cmp.astype('Int64'), how='left', on=['EncounterKey','OrderedDateKey', 'OrderedTimeOfDay'])
#fill 0 for those with no med/cmp
cmp_columns = [c for c in final_cases.columns if (c.startswith('cmp_'))]
final_cases[cmp_columns] =final_cases[cmp_columns].fillna(value=0)

### Medications

In [91]:
patients = ",".join([str(x) for x in final_cases['PatientDurableKey'].unique()])
medications = send_query(f"SELECT MedicationKey, StartDateKey AS MedStartDateKey, EncounterKey AS MedEncounterKey, PatientDurableKey \
FROM CAB_EPIC.dbo.MedicationEventFact WHERE PatientDurableKey in ({patients})")

In [92]:
medications = medications.merge(send_query("SELECT MedicationKey, SimpleGenericName As MedName, PharmaceuticalSubclass as MedClass FROM CAB_EPIC.dbo.MedicationDim"),
                                how='inner', left_on='MedicationKey', right_on='MedicationKey')

In [93]:
len(medications['MedClass'].unique())

694

In [94]:
# Merge on patientkey for historical medications 
patients_rx = final_cases[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay']].merge(medications, how='left', left_on='PatientDurableKey', right_on='PatientDurableKey')

In [95]:
# filter for medication before the test
patients_rx = patients_rx[patients_rx['MedStartDateKey'] < patients_rx['OrderedDateKey']]
# and make sure they are not from same encounter (just to be extra confident)
patients_rx = patients_rx[patients_rx['MedEncounterKey'] != patients_rx['EncounterKey']]

In [97]:
# Pivot on encounter key, since duplicate patients, but even if from same encounter medical information should be same regardless
patients_rx = patients_rx[['EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'MedClass']]
patients_rx = patients_rx.pivot_table(index=['EncounterKey','OrderedDateKey', 'OrderedTimeOfDay'], columns='MedClass', aggfunc=len, fill_value=0)
patients_rx = patients_rx.add_prefix('rx_')

In [99]:
final_cases = final_cases.merge(patients_rx.astype('Int64'), how='left', on=['EncounterKey','OrderedDateKey', 'OrderedTimeOfDay'])

In [100]:
rx_columns = [c for c in final_cases.columns if (c.startswith('rx_'))]
final_cases[rx_columns] = final_cases[rx_columns].fillna(value=0)

In [101]:
final_cases.to_csv('/datafs_a/carolgao/haim14_data/blood_infection/base_full_tabular.csv')

### Notes

In [83]:
final_cases = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/base_full_tabular.csv")

In [122]:
# Drop encounters with not enough past history info
rx_columns = [c for c in final_cases.columns if (c.startswith('rx_'))]
icd_columns = [c for c in final_cases.columns if (c.startswith('icd_'))]
cmp_columns = [c for c in final_cases.columns if (c.startswith('cmp_'))]
final_cases['count_rx'] = (final_cases[rx_columns].sum(axis=1) > 0).astype(int)
final_cases['count_icd'] = (final_cases[icd_columns].sum(axis=1) > 0).astype(int)
final_cases['count_cmp'] = (final_cases[cmp_columns].sum(axis=1) >0).astype(int)

final_cases = final_cases.loc[final_cases['count_rx'] + final_cases['count_icd'] + final_cases['count_cmp'] >= 2]

In [124]:
encounters =",".join([str(x) for x in final_cases['EncounterKey'].unique()])

In [10]:
send_query(f"select distinct Type from CAB_EPIC.dbo.ClinicalNoteFact cnf where cnf.Type like '%ED %'")

,Type
0,CLC Shared Information
1,ED AVS Snapshot
2,Performed Procedure
3,Scanned Notes
4,ED Update
5,ED Observation Discharge
6,ED Triage Notes
7,ED Follow Up Summary
8,ED Front end provider
9,ED Procedures


In [129]:
%%time
notes_raw = send_query(f"select distinct cnf.PatientDurableKey, cnf.EncounterKey, cnf.Service as OtherNotesService, cnf.ClinicalNoteKey, cnt.Text as OtherNotes, \
cnf.Type as OtherNotesType, cnf.ServiceDateKey as OtherNotesDate, cnf.ServiceTimeOfDayKey as OtherNotesTime from CAB_EPIC.dbo.ClinicalNoteFact cnf \
inner join CAB_EPIC.dbo.ClinicalNoteTextFact cnt on cnf.ClinicalNoteKey = cnt.ClinicalNoteKey \
where cnf.Type in ('H&P (View-Only)', 'H&P', 'Initial Assessments', 'AdmissionNote') \
and cnf.EncounterKey in ({encounters}) ")

CPU times: user 1.72 s, sys: 2.49 s, total: 4.21 s
Wall time: 9min 33s


In [130]:
notes_raw.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/hp_notes_raw.csv", index = False)

In [62]:
final_cases = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/base_with_notes.csv")
final_cases = final_cases.drop(columns = ['CombinedNotes'])

In [30]:
other_notes = final_cases.merge(notes_raw, how="inner", left_on=['PatientDurableKey','EncounterKey'], right_on=['PatientDurableKey','EncounterKey'])

In [31]:
# Restrcit to before lab was ordered
other_notes = other_notes.loc[(other_notes['OtherNotesDate'] < other_notes['OrderedDateKey']) | 
            ((other_notes['OtherNotesDate'] == other_notes['OrderedDateKey']) 
           & (other_notes['OtherNotesTime'] <= other_notes['OrderedTimeOfDay']))]

### Restrict to most recent notes

In [33]:
other_notes['OtherNotesDateTime'] = pd.to_datetime(
    other_notes['OtherNotesDate'].astype(str) + 
    other_notes['OtherNotesTime'].astype(str).str.zfill(4),
    format='%Y%m%d%H%M'
)

# Sort descending by datetime
other_notes = other_notes.sort_values(['PatientDurableKey', 'EncounterKey', 'OtherNotesDateTime'], ascending=[True, True, False])

In [49]:
# Take the most recent 1 per group
top1 = (
    other_notes
    .groupby(['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'])
    .head(1)
)

# Combine 'OtherNotes' within those top 3 per group
top1['CombinedNotes'] = (
    top3.groupby(['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'])['OtherNotes']
        .transform(lambda x: ' '.join(x))
)

# If you only want one combined row per group:
latest1 = (
    top1.groupby(['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], as_index=False)
        .agg({'CombinedNotes': 'first'})
)


In [53]:
final_cases = final_cases.merge(latest1, how = "left", left_on=['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], right_on=['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'])
notes_matched = final_cases[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'CombinedNotes']]
notes_matched.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/notes_matched_most_recent_1.csv')

In [133]:
# Combine notes
other_notes['CombinedNotes'] = other_notes.groupby(['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'])['OtherNotes'].transform(lambda x : ' '.join(x)) 

In [134]:
# unique by encounter key and order time  
unique_patient_notes = other_notes[['PatientDurableKey','EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'CombinedNotes']].drop_duplicates()

In [135]:
final_cases = final_cases.merge(unique_patient_notes, how = "left", left_on=['PatientDurableKey','EncounterKey','OrderedDateKey','OrderedTimeOfDay'], right_on=['PatientDurableKey','EncounterKey','OrderedDateKey','OrderedTimeOfDay'])

In [136]:
# Save notes
notes_matched = final_cases[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'CombinedNotes']]
notes_matched.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/notes_matched.csv')

In [18]:
final_cases.to_csv(f'/datafs_a/haim14_data/blood_infection/base_with_notes.csv')

### Generate notes embeddings

In [ ]:
final_cases = pd.read_csv("/datafs_a/carolgao/haim14_data/blood_infection/base_with_notes.csv")

In [ ]:
# DO THIS IN A DIFFERENT PYTHON SCRIPT
from generate_embeddings import * 
generate_embeddings(cohort = "blood", spec = "CombinedNotes")

In [5]:
notes_embeddings = pd.read_csv('/datafs_a/carolgao/haim14_data/blood_infection/blood_processed_notes_CombinedNotes.csv')

In [11]:
#merge with cases 
final_cases = final_cases.merge(notes_embeddings, how = "left", left_on=['PatientDurableKey','EncounterKey','OrderedDateKey','OrderedTimeOfDay'], right_on=['PatientDurableKey','EncounterKey','OrderedDateKey','OrderedTimeOfDay'])

In [18]:
final_cases.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_tabular_notes.csv')

## Extract Hospital 

In [56]:
def find_locations(encounters): 
    locations = send_query(f"select distinct d1.EncounterKey, d2.LocationName from CAB_EPIC.dbo.PatientLocationEventFact d1 \
    INNER JOIN CAB_EPIC.dbo.DepartmentDim d2 on d1.DepartmentKey = d2.DepartmentKey \
    where d1.EncounterKey in ({encounters})" )
    locations = locations.loc[(locations['LocationName'] != "*Not Applicable") & (locations['LocationName'] != "*Unknown") & (locations['LocationName'] != "*Unspecified")]
    locations['Hospital'] = ""
    locations.loc[(locations['LocationName'].str.startswith("HHC BH")) | (locations['LocationName'].str.startswith("BH")), 'Hospital'] = "BH"
    locations.loc[(locations['LocationName'].str.startswith("HHC HH")) | (locations['LocationName'].str.startswith("HH ")), 'Hospital'] = "HH"
    locations.loc[(locations['LocationName'].str.startswith("HHC HOCC")) | (locations['LocationName'].str.startswith("HOCC")) | (locations['LocationName'] == 'HHC The Hospital of Central Connecticut Parent'), 'Hospital'] = "HOCC"
    locations.loc[(locations['LocationName'].str.startswith("HHC CH")) | (locations['LocationName'].str.startswith("CH")), 'Hospital'] = "CH"
    locations.loc[(locations['LocationName'].str.startswith("HHC SV")) | (locations['LocationName'].str.startswith("SV")), 'Hospital'] = "SV"
    locations.loc[(locations['LocationName'].str.startswith("HHC MMC")) | (locations['LocationName'].str.startswith("MMC")), 'Hospital'] = "MMC"
    locations.loc[(locations['LocationName'].str.startswith("HHC WH")) | (locations['LocationName'].str.startswith("WH")), 'Hospital'] = "WH"
    locations.loc[(locations['LocationName'].str.startswith("SBO")), 'Hospital'] = "SBO"
    locations = locations[['EncounterKey', 'Hospital']].drop_duplicates()
    return locations 

In [20]:
encounters =",".join([str(x) for x in final_cases['EncounterKey'].unique()])
locations = find_locations(encounters)

In [21]:
final_cases = pd.merge(final_cases, locations, how = 'left', on = ['EncounterKey'])

In [38]:
final_cases['Hospital'].value_counts()

Hospital
HH      43594
BH      22190
HOCC    21727
SV      16842
MMC     15284
CH      11785
WH       7622
           10
SBO         2
Name: count, dtype: int64

In [40]:
base_HH = final_cases.loc[final_cases['Hospital'] == "HH"]
base_HOCC = final_cases.loc[final_cases['Hospital'] == "HOCC"]
base_BH = final_cases.loc[final_cases['Hospital'] == "BH"]
base_MMC = final_cases.loc[final_cases['Hospital'] == "MMC"]

In [41]:
base_HH.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_HH_recent1.csv')
base_HOCC.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_HOCC_recent1.csv')
base_BH.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_BH_recent1.csv')
base_MMC.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_MMC_recent1.csv')

In [42]:
final_cases.groupby(['OrderedYear']).agg(
    positive_rate=('Positive', 'mean'),
    N = ('Positive', 'count')
)

,positive_rate,N
OrderedYear,,
2022,0.087623,41245
2023,0.085649,41997
2024,0.090728,36648
2025,0.092816,19167


In [43]:
base_HH.groupby(['OrderedYear']).agg(
    positive_rate=('Positive', 'mean'),
    N = ('Positive', 'count')
)

,positive_rate,N
OrderedYear,,
2022,0.087471,13273
2023,0.083707,13165
2024,0.087422,11576
2025,0.090681,5580
